# Amazon Bedrock AgentCore Observability: Proteção de Dados

À medida que as organizações adotam cada vez mais sistemas de IA agêntica para automatizar fluxos de trabalho complexos e processos de tomada de decisão, proteger dados sensíveis tornou-se uma preocupação crítica. Agentes de IA frequentemente lidam com informações de identificação pessoal (PII), dados financeiros, registros de saúde e outras informações confidenciais que devem ser protegidas ao longo de todo o ciclo de vida do agente, desde o processamento de entrada até a geração de saída.

Este notebook demonstra uma abordagem abrangente para proteger dados sensíveis em aplicações de IA agêntica, combinando Amazon Bedrock Guardrails e políticas de Proteção de Dados do Amazon CloudWatch Logs. Embora estejamos hospedando nosso agente no runtime do AgentCore usando o framework Strands para esta demonstração, esses princípios e técnicas de proteção de dados podem ser aplicados a agentes hospedados em qualquer runtime e qualquer framework, tornando os conceitos agnósticos de plataforma e adaptáveis à sua infraestrutura de agentes existente.


# O Que Você Aprenderá
Neste tutorial prático, exploraremos:

- Como detectar informações sensíveis nas interações do Agente e nos Logs e Traces do CloudWatch
- Amazon Bedrock Guardrails: Como configurar filtros de informações sensíveis para impedir que agentes de IA processem ou gerem conteúdo sensível
- Proteção de Dados do CloudWatch Logs: Como detectar e mascarar automaticamente dados sensíveis nos logs de aplicação, garantindo que PII e outras informações confidenciais não vazem através dos mecanismos de logging
- Integração com AgentCore: Como implementar essas medidas de proteção dentro de fluxos de trabalho agênticos, criando uma estratégia de defesa em profundidade para suas aplicações de IA

# Arquitetura
O diagrama abaixo ilustra a arquitetura de alto nível para este tutorial, que inclui:

- Um Agente de IA construído usando o SDK Strands e hospedado no Runtime do AgentCore
- Integração com a Observabilidade do AgentCore, onde sinais de telemetria são capturados no Amazon CloudWatch e visualizados usando painéis de Observabilidade GenAI, logs e traces

<div style="text-align:left">
    <img src="images/agentcore_observability_data_protection_architecture.png" width="100%"/>
</div>


# Por Que Isso Importa

Sem as devidas salvaguardas, sistemas de IA agêntica podem:

- Expor inadvertidamente dados sensíveis de clientes em respostas ou logs
- Processar ou reter informações que violam regulamentações de privacidade (LGPD, GDPR, HIPAA, CCPA)
- Gerar saídas contendo PII que não deveriam ser compartilhadas
- Criar vulnerabilidades de conformidade e segurança na infraestrutura da sua aplicação

Ao implementar Bedrock Guardrails e Proteção de Dados do CloudWatch Logs juntos, você cria múltiplas camadas de proteção que trabalham em conjunto para proteger suas aplicações de IA agêntica desde a entrada até a saída e o logging.


# Visualizar dados de observabilidade dos seus agentes Amazon Bedrock AgentCore

Após implementar a observabilidade no seu agente, você pode [visualizar os logs, métricas e traces coletados](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-view.html) tanto na página de observabilidade de IA generativa do console do CloudWatch quanto no CloudWatch Logs. Para saber mais sobre o uso da observabilidade de IA generativa no CloudWatch, incluindo como visualizar os dados de sessão e trace individuais dos seus agentes, consulte [Agentes Amazon Bedrock AgentCore](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/AgentCore-Agents.html) no guia do usuário do Amazon CloudWatch.

`Os log groups de agentes AgentCore têm o seguinte formato`:

`1. Logs Padrão`

- Formato de logs padrão: saída stdout/stderr
- Localização: /aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint_name>/[runtime-logs] <UUID>
- Contém: Erros de runtime, logs de aplicação, instruções de depuração

Exemplo de Uso:
- print("Processando requisição...") # Aparece nos logs padrão
- logging.info("Requisição processada com sucesso") # Aparece nos logs padrão


`2. Logs estruturados OTEL - Informações detalhadas de operação`

- Localização: /aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint_name>/otel-rt-logs
- Contém: Detalhes de execução, rastreamento de erros, dados de desempenho
- Coleta automática: Nenhum código adicional necessário - gerado pela instrumentação ADOT
- Benefícios: Pode incluir IDs de correlação vinculando logs a traces relevantes

`3. Traces e Spans`

Traces fornecem visibilidade nos caminhos de execução de requisições através do seu agente:

- Localização: /aws/spans/default
- Acesso via: Console do CloudWatch Transaction Search
- Requisitos: CloudWatch Transaction Search deve estar habilitado

Traces capturam automaticamente:
- Sequências de invocação do agente
- Integração com componentes de framework (LangChain, etc.)
- Chamadas e respostas de LLM
- Invocações e resultados de ferramentas
- Caminhos de erro e exceções


<span style="color:red;">Para este laboratório, focaremos na proteção dos logs padrão.</span>

# Pré-requisitos

- Habilitar a pesquisa de transações no Amazon CloudWatch. Usuários de primeira vez devem [habilitar o CloudWatch Transaction Search](../../00-enable-transaction-search-template/enable_transaction_search.ipynb) para visualizar spans e traces do Bedrock AgentCore.
- Conta AWS com acesso ao modelo Amazon Bedrock Claude Haiku 4.5 com Model ID: global.anthropic.claude-haiku-4-5-20251001-v1:0
- Credenciais AWS configuradas usando aws configure
- Conceder permissões IAM apropriadas necessárias para criar ou trabalhar com uma política de proteção de dados [documentação](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/data-protection-policy-permissions.html), Bedrock Guardrails e AgentCore


# 1. Instalação e configuração

Instale as dependências necessárias:


In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet


# 2. Criar um Agente sem proteção de dados habilitada

Vamos executar um Agente primeiro sem nenhuma proteção de dados habilitada e examinar os resultados. Para isso, estamos usando um conjunto de dados de exemplo de uma interação de um Agente de Central de Atendimento com seu Cliente, que contém informações sensíveis. Você pode examinar o conteúdo [aqui](./data/customer_support_conversation_sample.txt). Atualizamos o prompt do Agente para resumir a conversa nos dados de exemplo fornecidos.

Também adicionamos informações sensíveis nas instruções `print` conforme abaixo:

            "agent.type": "customer_agent_reviewer",
            "agent.email": "jrussell@domain.com",
            "agent.phone": "301-555-0100",
            "agent.id": "ABCDE12345"

Intencionalmente instruímos o Agente a revelar informações sensíveis conforme abaixo:

            user_input = """summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email?"""



In [ ]:
%%writefile data_protection.py
import os
import logging
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configure Strands logging
logging.getLogger("strands").setLevel(logging.INFO)

app = BedrockAgentCoreApp()

@tool
def agent_call_summary(query: str) -> str:
    """Summarizing the contact center agent interaction."""
    
    try:
        logger.info(f"Processing agent call summary for query: {query[:50]}...")
        
        # Read the customer support conversation data
        results = open('./data/customer_support_conversation_sample.txt', 'r').read()
        
        logger.info(f"Agent conversation search completed successfully for query: {query[:50]}...")       
        return results
        
    except Exception as e:
        logger.error(f"Agent call summary failed: {str(e)}")
        return f"Search error: {str(e)}"

def get_bedrock_model():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
        
    try:
        bedrock_model = BedrockModel(
            model_id=model_id,
            temperature=0.7,
            max_tokens=1028         
        )
        logger.info(f"Successfully initialized Bedrock model: {model_id}")
        return bedrock_model
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock model: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

# Initialize the model and agent
bedrock_model = get_bedrock_model()


# Create customer support agent
support_agent = Agent(
    model=bedrock_model,
    system_prompt="""You are an expert customer support conversation agent specializing in finding 
                     accurate and relevant information. Your role is to efficiently search, analyze, and synthesize
                     information provided to answer user queries comprehensively. You should provide
                     well-researched responses with current information, clear summaries, and cite reliable sources
                     when presenting your findings. If a user asks to perform a task that can be accomplished with a tool, 
                     you must use the tool. You have access to the agent_call_summary tool which already has data and summarizes 
                     the call center support agent conversation. If user doesnt provide specific conversation or agent details,
                     always default to the agent Jane Doe and use agent_call_summary tool's summarization.""",
    tools=[agent_call_summary],
    trace_attributes={        
        "tags": ["Strands", "Observability", "CustomerSupport"]
    }
)

@app.entrypoint
def customer_support_agent(payload):
    """Invoke the customer support agent with a payload"""
    try:
        # Extract user input from payload
        user_input = payload.get("prompt", "")
        
        logger.info(f"User input: {user_input[:100]}...")

        print("agent.type: customer_agent_reviewer")
        print("agent.email: jrussell@domain.com")
        print("agent.phone: 301-555-0100")
        print("agent.id: ABCDE12345")
        
        # If no specific query provided, use default
        if not user_input:
            user_input = "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email."
        
        # Execute the customer research task
        response = support_agent(user_input)
        
        logger.info("Agent response generated successfully")
        return response.message['content'][0]['text']
        
    except Exception as e:
        logger.error(f"Agent execution failed: {str(e)}")
        return f"Error processing request: {str(e)}"

if __name__ == "__main__":
    app.run()


# Implantar o agente no Runtime do AgentCore


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "customer_support_agent"
response = agentcore_runtime.configure(
    entrypoint="data_protection.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY'
)
response

# Iniciando o agente no Runtime do AgentCore

In [ ]:
launch_result = agentcore_runtime.launch()

# Invocando o Runtime do AgentCore

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email."})
invoke_response



Vamos revisar os resultados começando pela interação do Agente aqui. Nota: Suas respostas geradas podem ser diferentes, mas o conceito ainda se aplica! Recomendamos fortemente que você leia o arquivo de texto com dados de exemplo e improvise os prompts adequadamente para invocar o agente/ferramenta, já que o comportamento do modelo muda frequentemente.)


<div style="text-align:left">
    <img src="images/agent_response_without_data_protection.png" width="100%"/>
</div>



Trace do CloudWatch:


<div style="text-align:left">
    <img src="images/trace_without_data_protection.png" width="100%"/>
</div>


Logs do CloudWatch (logs de runtime do agente):


<div style="text-align:left">
    <img src="images/logs_without_data_protection.png" width="100%"/>
</div>





# 3. Habilitar Bedrock Guardrails

Guardrails para Amazon Bedrock avalia entradas de usuários e respostas de modelos fundacionais com base em políticas específicas de caso de uso, e fornece uma camada adicional de salvaguardas independentemente do modelo fundacional subjacente. Guardrails podem ser aplicados em todos os grandes modelos de linguagem (LLMs) no Amazon Bedrock, incluindo modelos com fine-tuning. Os clientes podem criar múltiplos guardrails, cada um configurado com uma combinação diferente de controles, e usar esses guardrails em diferentes aplicações e casos de uso.

Você pode usar Amazon Bedrock Guardrails de múltiplas formas para ajudar a proteger suas aplicações de IA generativa. Por exemplo:

- Uma aplicação de chatbot pode usar guardrails para ajudar a filtrar entradas prejudiciais de usuários e respostas tóxicas do modelo.
- Uma aplicação bancária pode usar guardrails para ajudar a bloquear consultas de usuários ou respostas do modelo associadas à busca ou fornecimento de consultoria de investimentos.
- Uma aplicação de central de atendimento para resumir transcrições de conversas entre usuários e agentes pode usar guardrails para redigir informações de identificação pessoal (PII) dos usuários para proteger a privacidade.

Guardrails para Amazon Bedrock possuem múltiplos componentes que incluem Filtros de Conteúdo, Tópicos Negados, Filtros de Palavras e Frases, e Filtros de Informações Sensíveis (PII, PHI). Para uma lista completa, consulte a [documentação](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html).

Para este exercício, focaremos apenas na proteção de Informações Sensíveis.

Crie um Bedrock Guardrail conforme abaixo. Para demonstrar guardrails, anonimizamos as informações sensíveis, mas você pode optar por `BLOQUEAR` completamente os prompts/respostas. Para conhecer todos os filtros de informações sensíveis disponíveis, consulte a [documentação](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-sensitive-filters.html).




In [ ]:
import boto3
bedrock_client = boto3.client('bedrock')

create_response = bedrock_client.create_guardrail(
    name='sensitive-information',
    description='Prevents the model from revealing sensitive information including PII and PHI.',
    sensitiveInformationPolicyConfig={
        'piiEntitiesConfig': [
            {
                'type': 'EMAIL',
                'action': 'ANONYMIZE'
            },
            {
                'type': 'PHONE',
                'action': 'ANONYMIZE'
            },
            {
                'type': 'NAME',
                'action': 'ANONYMIZE',
                'inputAction': 'NONE'
            },
            {
                'type': 'US_SOCIAL_SECURITY_NUMBER',
                'action': 'ANONYMIZE'
            },
            {
                'type': 'US_BANK_ACCOUNT_NUMBER',
                'action': 'ANONYMIZE'
            },
            {
                'type': 'CREDIT_DEBIT_CARD_NUMBER',
                'action': 'ANONYMIZE'
            }
        ],
        'regexesConfig': [
            {
                'name': 'Account Number',
                'description': 'Matches account numbers in the format XXXXXX1234',
                'pattern': r'\b\d{6}\d{4}\b',
                'action': 'ANONYMIZE'
            }
        ]
    },
    blockedInputMessaging='Sorry, guardrails intervened and model cannot answer the question.',
    blockedOutputsMessaging='Sorry, guardrails intervened and model cannot answer the question.',
)

print(create_response)
guardrailId = create_response['guardrailId']



Crie uma nova versão do guardrail para usar com o Agente.


In [ ]:
version_response = bedrock_client.create_guardrail_version(
    guardrailIdentifier=guardrailId,
    description='Version of Guardrail that has HIGH content filters across'
)
guardrail_version_number = version_response['version']

print(guardrail_version_number)
print(version_response)




Vamos aplicar o Guardrail recém-criado ao Agente conforme abaixo:

            guardrail_id=os.getenv("BEDROCK_GUARDRAIL_ID"),      # Seu ID do Bedrock guardrail
            guardrail_version=os.getenv("BEDROCK_GUARDRAIL_VERSION"),                   # Versão do Guardrail
            guardrail_trace="enabled",               # Habilitar informações de trace para depuração

Para saber mais sobre como aplicar guardrails a Agentes Strands, consulte a [documentação](https://strandsagents.com/latest/documentation/docs/user-guide/safety-security/guardrails/).

Nota: Para sua conveniência, neste laboratório, usamos o mesmo guardrail id e número de versão criados acima. Mas você pode substituir pelo seu guardrailId e versão específicos conforme necessário.



In [ ]:
%%writefile data_protection.py
import os
import logging
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configure Strands logging
logging.getLogger("strands").setLevel(logging.INFO)

app = BedrockAgentCoreApp()



@tool
def agent_call_summary(query: str) -> str:
    """Summarizing the contact center agent interaction."""
    try:
        logger.info(f"Processing agent call summary for query: {query[:50]}...")
        
        # Read the customer support conversation data
        results = open('./data/customer_support_conversation_sample.txt', 'r').read()
        
        logger.info(f"Agent conversation search completed successfully for query: {query[:50]}...")
        return results
        
    except Exception as e:
        logger.error(f"Agent call summary failed: {str(e)}")
        return f"Search error: {str(e)}"

def get_bedrock_model():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
    
    logger.info(f"BEDROCK_GUARDRAIL_ID: {os.getenv('BEDROCK_GUARDRAIL_ID')}")
    logger.info(f"BEDROCK_GUARDRAIL_VERSION: {os.getenv('BEDROCK_GUARDRAIL_VERSION')}")

    
    try:
        bedrock_model = BedrockModel(
            model_id=model_id,
            temperature=0.7,
            max_tokens=1028,
            guardrail_id=os.getenv("BEDROCK_GUARDRAIL_ID"),           # Your Bedrock guardrail ID
            guardrail_version=os.getenv("BEDROCK_GUARDRAIL_VERSION"),                        # Guardrail version
            guardrail_trace="enabled",                    # Enable trace info for debugging            
        )        
        logger.info(f"Successfully initialized Bedrock model: {model_id}")
        return bedrock_model
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock model: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

# Initialize the model and agent
bedrock_model = get_bedrock_model()

# Create customer support agent
support_agent = Agent(
    model=bedrock_model,
    system_prompt="""You are an expert customer support conversation agent specializing in finding 
                     accurate and relevant information. Your role is to efficiently search, analyze, and synthesize
                     information provided to answer user queries comprehensively. You should provide
                     well-researched responses with current information, clear summaries, and cite reliable sources
                     when presenting your findings. If a user asks to perform a task that can be accomplished with a tool, 
                     you must use the tool. You have access to the agent_call_summary tool which already has data and summarizes 
                     the call center support agent conversation. If user doesnt provide specific conversation or agent details,
                     always default to the agent Jane Doe and use agent_call_summary tool's summarization.""",
    tools=[agent_call_summary],
    trace_attributes={
        "tags": ["Strands", "Observability", "CustomerSupport"]
    }
)

@app.entrypoint
def customer_support_agent(payload):
    """Invoke the customer support agent with a payload"""
    try:
        # Extract user input from payload
        user_input = payload.get("prompt", "")
        
        logger.info(f"User input: {user_input[:100]}...")

        print("agent.type: customer_agent_reviewer")
        print("agent.email: jrussell@domain.com")
        print("agent.phone: 301-555-0100")
        print("agent.id: ABCDE12345")
        
        # If no specific query provided, use default
        if not user_input:
            user_input = "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email."
        
        # Execute the customer research task
        response = support_agent(user_input)
        
        logger.info("Agent response generated successfully")
        return response.message['content'][0]['text']
        
    except Exception as e:
        logger.error(f"Agent execution failed: {str(e)}")
        return f"Error processing request: {str(e)}"

if __name__ == "__main__":
    app.run()


Agora, reinicie o agente


# Iniciando o agente no Runtime do AgentCore

In [ ]:
launch_result

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_GUARDRAIL_ID": guardrailId, 
        "BEDROCK_GUARDRAIL_VERSION": guardrail_version_number  
    }
)
launch_result

# Testar o agente novamente

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email."})
invoke_response


Vamos revisar os resultados com Guardrails aplicados.

Você pode ver que as informações sensíveis estão 'anonimizadas' conforme as configurações dos guardrails. Note que não incluímos `endereço` nos guardrails e, portanto, o endereço ainda é exibido.


<div style="text-align:left">
    <img src="images/agent_response_with_bedrock_guardrails.png" width="100%"/>
</div>



Informações de Trace do CloudWatch abaixo com informações sensíveis anonimizadas (exceto endereço, que não incluímos nos guardrails).

<div style="text-align:left">
    <img src="images/trace_with_bedrock_guardrails.png" width="100%"/>
</div>




Observe abaixo que as informações sensíveis nos logs (das instruções `print`) ainda estão expostas.


<div style="text-align:left">
    <img src="images/logs_with_guardrails_no_logs_data_protection.png" width="100%"/>
</div>






# 4. Habilitar Proteção de Dados do CloudWatch Logs

Guardrails podem ajudar a proteger informações sensíveis nos prompts e respostas do agente. A [proteção de dados do CloudWatch Logs](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/mask-sensitive-log-data.html) pode ajudar a detectar e mascarar informações sensíveis nos logs. Combinar ambas as funcionalidades pode ajudar a fornecer proteção em camadas.

Para nosso exemplo, criaremos uma política de proteção de dados do CloudWatch Logs com alguns identificadores de dados gerenciados, incluindo Email, telefone, nome, número de seguro social, número de conta bancária e número de cartão de crédito. Também incluímos Identificadores de Dados Personalizados (CDIs), que permitem definir suas próprias expressões regulares personalizadas que podem ser usadas na sua política de proteção de dados. Usando identificadores de dados personalizados, você pode atender casos de uso de informações de identificação pessoal (PII) específicos do negócio que os identificadores de dados gerenciados não conseguem fornecer. Por exemplo, você pode usar um identificador de dados personalizado para procurar IDs de funcionários específicos da empresa. Identificadores de dados personalizados podem ser usados em conjunto com identificadores de dados gerenciados. Para a lista completa de tipos de dados que você pode proteger, consulte nossa [documentação](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/protect-sensitive-log-data-types.html).


Vamos habilitar a política de proteção de dados para este log group do Runtime do Agente. Opcionalmente, se necessário, você também pode habilitar a política de proteção de dados para o log group `aws/spans` conforme suas necessidades.



In [ ]:
import json
cloudwatch_logs_client = boto3.client('logs')

log_group_name = '/aws/bedrock-agentcore/runtimes/' + launch_result.agent_id + '-DEFAULT'

try:
    response = cloudwatch_logs_client.put_data_protection_policy(
        logGroupIdentifier=log_group_name,        
        policyDocument=json.dumps(json.load(open('./cloudwatch_data_protection_policy.json')))
    )
    print("Data protection policy applied successfully:")
    print(response)
except cloudwatch_logs_client.exceptions.ResourceNotFoundException:
    print(f"Error: Log group '{log_group_name}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")




Você pode verificar que a proteção de dados está habilitada por:

- Faça login no console do CloudWatch
- Log Groups
- Selecione o log group do runtime do seu agente (você pode obter seu grupo na resposta acima para 'logGroupIdentifier') e escolha a aba 'Data protection' conforme abaixo:


<div style="text-align:left">
    <img src="images/cloudwatch_logs_after_data_protection_enabled.png" width="100%"/>
</div>




Agora, teste o agente novamente com guardrails e proteção de dados habilitados

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "summarize the agent conversation for doejane. Tell me exactly the phone number, name, address, and email?"})
invoke_response

Revise os resultados com guardrails e proteção de dados de logs habilitados.

Interação com o agente abaixo (sua saída exata pode variar, mas o conceito ainda se aplica):


<div style="text-align:left">
    <img src="images/agent_response_with_data_protection_enabled.png" width="100%"/>
</div>




Agora podemos ver que as informações sensíveis nos atributos do trace também estão protegidas. Observe também que o 'agent.id' está mascarado, o que se baseia no Identificador de Dados Personalizado com Regex.


<div style="text-align:left">
    <img src="images/logs_with_data_protection_enabled.png" width="100%"/>
</div>





# 5. Limpeza (Opcional)




In [ ]:
response = bedrock_client.delete_guardrail(
    guardrailIdentifier=guardrailId   # GUARDRAILID
)

In [ ]:
cloudwatch_response = cloudwatch_logs_client.delete_data_protection_policy(
    logGroupIdentifier=log_group_name
)

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
    
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)




# 6. Conclusão

**Principais Aprendizados**
Parabéns! Você aprendeu com sucesso como implementar medidas abrangentes de proteção de dados para seus agentes. Vamos recapitular o que abordamos:

O Que Realizamos

✅ Configuramos Bedrock Guardrails para:

- Redigir informações sensíveis
- Aplicar guardrails aos seus agentes

✅ Habilitamos a Proteção de Dados do CloudWatch Logs para:

- Detectar e mascarar automaticamente dados sensíveis nos logs
- Implementar identificadores de dados para dados sensíveis e padrões personalizados

✅ Integramos ambas as funcionalidades perfeitamente com o Bedrock AgentCore para implantações prontas para produção


**Melhores Práticas**

- Camadas de defesa: Use tanto guardrails (proteção em tempo de execução) quanto proteção de dados de logs (segurança pós-processamento)
- Teste rigorosamente: Valide suas políticas de guardrail com casos de teste diversos antes da implantação em produção
- Monitore e itere: Revise regularmente as métricas do CloudWatch e os logs de auditoria para refinar suas configurações
- Princípio do menor privilégio: Garanta que as roles IAM tenham apenas as permissões necessárias para guardrails e logging
- Documente suas políticas: Mantenha documentação clara sobre qual conteúdo é filtrado e por quê


**Próximos Passos**
Para aprimorar ainda mais a segurança do seu agente de IA:

- Explore filtros de palavras personalizadas e padrões regex para terminologia específica do setor
- Implemente testes A/B com diferentes configurações de guardrail
- Configure alarmes do CloudWatch para métricas de intervenção de guardrails
- Considere criptografia AWS KMS para seus log groups contendo operações sensíveis


**Recursos Adicionais**

- [Achados de auditoria de proteção de dados do CloudWatch Logs](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/mask-sensitive-log-data-audit-findings.html)
- [Política em nível de conta](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/mask-sensitive-log-data-accountlevel.html)
- [Casos de uso de Bedrock Guardrails](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-use.html)


**Lembre-se**: IA Responsável não é uma configuração única — é um compromisso contínuo. Continue a monitorar, avaliar e melhorar suas salvaguardas à medida que seus agentes evoluem e novas ameaças surgem.

Bom desenvolvimento, e mantenha-se seguro! 🛡️🤖